In [54]:
import pandas as pd

import inria_course_ml.common_mod as cmm
from sklearn.compose import make_column_selector, ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.ensemble import HistGradientBoostingClassifier

**USING NUMERICAL AND CATEGORICAL VARIABLES TOGETHER**

In [55]:
census_B = cmm.dfFromArffFile('/data/phpMawTba.arff')

Metadata =============================================
 Dataset: adult
	age's type is numeric
	workclass's type is nominal, range is ('Private', 'Self-emp-not-inc', 'Self-emp-inc', 'Federal-gov', 'Local-gov', 'State-gov', 'Without-pay', 'Never-worked')
	fnlwgt's type is numeric
	education's type is nominal, range is ('Bachelors', 'Some-college', '11th', 'HS-grad', 'Prof-school', 'Assoc-acdm', 'Assoc-voc', '9th', '7th-8th', '12th', 'Masters', '1st-4th', '10th', 'Doctorate', '5th-6th', 'Preschool')
	education-num's type is numeric
	marital-status's type is nominal, range is ('Married-civ-spouse', 'Divorced', 'Never-married', 'Separated', 'Widowed', 'Married-spouse-absent', 'Married-AF-spouse')
	occupation's type is nominal, range is ('Tech-support', 'Craft-repair', 'Other-service', 'Sales', 'Exec-managerial', 'Prof-specialty', 'Handlers-cleaners', 'Machine-op-inspct', 'Adm-clerical', 'Farming-fishing', 'Transport-moving', 'Priv-house-serv', 'Protective-serv', 'Armed-Forces')
	relationshi

In [56]:
target_nm = 'class'
target = census_B[target_nm]
features = census_B.drop(columns=[target_nm, 'fnlwgt', 'education-num'])
features.columns

Index(['age', 'workclass', 'education', 'marital-status', 'occupation',
       'relationship', 'race', 'sex', 'capital-gain', 'capital-loss',
       'hours-per-week', 'native-country'],
      dtype='object')

Selection based on data types.

In [57]:
categorical_selector = make_column_selector(dtype_include='object')
numerical_selector = make_column_selector(dtype_exclude='object')
categ_cols = categorical_selector(features)
num_cols = numerical_selector(features)


Combined use of two different transformers.

In [58]:
num_scaler = StandardScaler()
categories_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
features_tranformer_A = ColumnTransformer([('numscaler', num_scaler, num_cols), ('hotencoder', categories_encoder, categ_cols)])

In [59]:
pipemodel = make_pipeline(features_tranformer_A, LogisticRegression(max_iter=500))
pipemodel

,steps,"[('columntransformer', ...), ('logisticregression', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('numscaler', ...), ('hotencoder', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


**Evaluation of the model with train-test split of the data.**

We do not need to make any manual preprocessing (calling the transform or fit_transform methods) as it is already handled when calling the predict method.

In [60]:
features_tr, features_test, target_tr, target_test = train_test_split(features, target, test_size=.25, random_state=13)
pipemodel.fit(features_tr, target_tr)

,steps,"[('columntransformer', ...), ('logisticregression', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('numscaler', ...), ('hotencoder', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [61]:
pd.DataFrame(pipemodel.predict(features_test)[:5]), target_test[:5]

(       0
 0  <=50K
 1  <=50K
 2  <=50K
 3  <=50K
 4  <=50K,
 19287    <=50K
 16606    <=50K
 16979    <=50K
 24846    <=50K
 26074    <=50K
 Name: class, dtype: object)

In [62]:
pipemodel.score(features_test, target_test)

0.8483334698222914

**Evaluation of the model with cross-validation**

In [63]:
cross_result = cross_validate(pipemodel,features,target, cv=5)
cross_result['test_score'].mean(), cross_result['test_score'].std()

(np.float64(0.8514803215540241), np.float64(0.002436145528969233))

**FITTING A MORE POWERFUL MODEL: gradient boosting trees.**

In [64]:
ord_categories_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
features_tranformer_B = ColumnTransformer([('ordencoder',ord_categories_encoder,categ_cols)], remainder='passthrough')
pipemodel_B = make_pipeline(features_tranformer_B, HistGradientBoostingClassifier())

In [65]:
%%time
pipemodel_B.fit(features_tr, target_tr)

CPU times: user 1.72 s, sys: 110 ms, total: 1.83 s
Wall time: 1.06 s


,steps,"[('columntransformer', ...), ('histgradientboostingclassifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('ordencoder', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [66]:
pipemodel_B.score(features_test, target_test)

0.8695438539022193

**EXERCISE M1.05**

No numerical scaling and integer-coded (ordinal) categories

In [67]:
%%time
cross_result_B = cross_validate(pipemodel_B, features, target)
cross_result_B['test_score'].mean(), cross_result_B['test_score'].std()

CPU times: user 12 s, sys: 775 ms, total: 12.8 s
Wall time: 7.19 s


(np.float64(0.8731829483006678), np.float64(0.00134328116011953))

Scaling numerical features

In [68]:
features_tranformer_C = ColumnTransformer([('numscaler', num_scaler, num_cols),('ordencoder',ord_categories_encoder,categ_cols)])
pipemodel_C = make_pipeline(features_tranformer_C, HistGradientBoostingClassifier())

In [69]:
%%time
cross_result_C = cross_validate(pipemodel_C, features, target)
cross_result_C['test_score'].mean(), cross_result_C['test_score'].std()


CPU times: user 11.2 s, sys: 692 ms, total: 11.9 s
Wall time: 6.55 s


(np.float64(0.8736743697512456), np.float64(0.002423024837433754))

One-hot encoding of categorical variables

In [70]:
# Without this configuration: OneHotEncoder(handle_unknown='ignore', sparse_output=False), it fails.
pipemodel_D = make_pipeline(features_tranformer_A, HistGradientBoostingClassifier())

In [71]:
%%time
cross_result_D = cross_validate(pipemodel_D, features, target)
cross_result_D['test_score'].mean(), cross_result_D['test_score'].std()

CPU times: user 32.6 s, sys: 3 s, total: 35.6 s
Wall time: 18.6 s


(np.float64(0.8732034631435799), np.float64(0.002238372472772568))